# SyftBox Manual E2E Test

This notebook walks through a complete end-to-end workflow between a **Data Owner (DO)** and a **Data Scientist (DS)** using SyftBox. It demonstrates dataset creation, browsing, job submission, code review, approval/rejection, execution, and result retrieval.

### Flow Overview

1. **Setup** — Configuration, authentication, and peer connection
2. **Data Owner** — Creates and publishes datasets
3. **Data Scientist** — Browses datasets, prototypes code on mock data, submits two jobs
4. **Data Owner** — Reviews submitted code, approves one job, rejects the other, and runs the approved job
5. **Data Scientist** — Retrieves the output of the approved job and the rejection reason for the other

### Prerequisites
- GCP project with **Google Drive API** enabled
- OAuth credentials JSON (`app_credentials.json`)
- DS token (`token_ds.json`)
- A `.env` file with: `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`

---

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

### 1.1 Configuration

Load environment variables and validate that credential files exist.

In [ ]:
import os
from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])

assert APP_CREDENTIALS.exists(), "App credentials missing"
assert TOKEN_DS.exists(), "DS credentials missing"

### 1.2 Create DO token (one-time)

Converts OAuth app credentials to a user-authorized Drive token.
Uncomment and run once — it will open a browser window for authorization.

In [ ]:
import syft_client as sc

do_token_path = str(APP_CREDENTIALS.parent / "token_do_email_test.json")
# do_token_path = sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=do_token_path,
#     do_scopes=True
# )
# print(f"DO token saved to: {do_token_path}")

### 1.3 Login

Log in as both the Data Owner and the Data Scientist. Each client maintains its own local SyftBox folder and syncs independently.

In [ ]:
do_client = sc.login_do(email=DO_EMAIL, token_path=str(do_token_path))

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

### 1.4 Optional — Wipe prior state

Uncomment to delete all local SyftBox data and start fresh.

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

### 1.5 Establish peer connection

The Data Scientist requests access to the Data Owner, and the Data Owner approves. This is required before any data or jobs can be exchanged.

In [ ]:
ds_client.add_peer(DO_EMAIL)

do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

---

## 2. Data Owner: Create Datasets

The Data Owner creates datasets that will be shared with the Data Scientist. Each dataset has:
- **Mock data** — a public, non-sensitive version that the DS can freely access and prototype against
- **Private data** — the real data that never leaves the DO's machine; only job code can access it at runtime

### 2.1 Create a simple text dataset

In [ ]:
do_client.create_dataset(
    name="testdata",
    mock_path="data/mock.txt",
    private_path="data/private.txt",
    users=[DS_EMAIL],
)

### 2.2 Create a PDF dataset

Generate sample PDFs, then create a dataset with a CSV manifest as mock data and the PDFs as private data.

In [ ]:
!uv run create_test_pdfs.py

In [ ]:
do_client.create_dataset(
    name="pdfdata",
    mock_path="data/manifest.csv",
    private_path="data/PDFs",
    summary="A dataset with some PDFs",
    readme_path="data/readme.md",
    tags=["test", "pdf"],
    users=[DS_EMAIL],
    upload_private=False,
    sync=True,
)

### 2.3 View datasets

The Data Owner can list all their datasets and inspect individual ones.

In [ ]:
do_client.datasets

In [ ]:
dataset = do_client.datasets.get("testdata", datasite=DO_EMAIL)
dataset

In [ ]:
mock_contents = dataset.mock_files[0].read_text()
print(f"Mock file contents: {mock_contents!r}")

---

## 3. Data Scientist: Browse Datasets and Prototype Code

The Data Scientist syncs to discover available datasets. They can only see mock data — the private data stays on the Data Owner's machine.

### 3.1 List available datasets

In [ ]:
ds_client.datasets

### 3.2 Write trial code against mock data

Before submitting a job, the Data Scientist can prototype their analysis locally using the mock data. The `resolve_dataset_file_path` function returns mock data when run locally, and private data when run as a job on the DO's machine.

> **Note:** Remove the `client=` parameter before submitting as a job — it will resolve automatically on the DO's machine.

In [ ]:
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata", client=ds_client)
df = pd.read_fwf(resolved_path)
print(f"Type of data in use: {df.columns[0]}")

### 3.3 Submit jobs

Now the Data Scientist submits two different jobs to the Data Owner. Each job is a self-contained Python script that will run in an isolated environment on the DO's machine.

- **Job A** (`summary_stats`) — computes basic statistics on the dataset
- **Job B** (`column_names`) — extracts and returns column names from the dataset

Both jobs use `resolve_dataset_file_path` to access the private data at runtime.

In [ ]:
import tempfile

def write_job_script(code: str) -> str:
    """Write a job script to a temp file and return its path."""
    job_dir = Path(tempfile.mkdtemp(prefix="syft_job_"))
    script = job_dir / "main.py"
    script.write_text(code)
    return str(script)

#### Job A: Summary Statistics

A simple analysis job that reads the dataset and computes summary statistics.

In [ ]:
JOB_A_NAME = "summary_stats"

job_a_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {
    "message": "Summary statistics computed successfully",
    "row_count": len(df),
    "columns": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_a_code,
    job_name=JOB_A_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_A_NAME}' submitted to {DO_EMAIL}")

#### Job B: Column Names

This job intentionally leaks column names from the private dataset — the kind of thing a Data Owner might want to reject.

In [ ]:
JOB_B_NAME = "column_names"

job_b_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

# This leaks sensitive column names from the private dataset
result = {
    "message": "Extracted column names",
    "column_names": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_b_code,
    job_name=JOB_B_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_B_NAME}' submitted to {DO_EMAIL}")

---

## 4. Data Owner: Review, Approve/Reject, and Run Jobs

The Data Owner syncs to receive the submitted jobs, reviews the code, and decides whether to approve or reject each one. Approved jobs are then executed on the DO's machine.

### 4.1 List incoming jobs

In [ ]:
do_client.jobs

### 4.2 Review the submitted code

Look up each job by name and inspect the code before making a decision.

In [ ]:
job_a = do_client.jobs[JOB_A_NAME]
print(f"--- {job_a.name} (status: {job_a.status}) ---")
print(job_a.code)

In [ ]:
job_b = do_client.jobs[JOB_B_NAME]
print(f"--- {job_b.name} (status: {job_b.status}) ---")
print(job_b.code)

### 4.3 View the full job object
Alternatively, you can view the full job object.

In [ ]:
job_a

### 4.3 Approve Job A, Reject Job B

The DO approves the summary statistics job and rejects the column names job because it would leak sensitive information.

In [ ]:
job_a.approve()

In [ ]:
job_b.reject(reason="The job leaks sensitive column names from the private dataset")

### 4.4 Run approved jobs

Execute all approved jobs. Each job runs in an isolated Python virtual environment on the DO's machine. The private dataset is accessible to the job code via `resolve_dataset_file_path`. You can pass these arguments: 
1. `stream_output`: If True (default), stream output in real-time. If False, capture output at end.
2. `timeout`: Timeout in seconds per job. Defaults to 300 (5 minutes).
3. `force_execution`: If True, process all approved jobs regardless of version compatibility. If False (default), skip jobs from peers with incompatible or unknown versions.
4. `share_outputs_with_submitter`: If True, grant read access on outputs to submitter.
5. `share_logs_with_submitter`: If True, grant read access on logs to submitter.

In [ ]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

---

## 5. Data Scientist: Retrieve Results

The Data Scientist syncs to pull back the job results. They can read the output of the approved job and see the reason for the rejected one.

### 5.1 Check job statuses

In [ ]:
ds_client.jobs

### 5.2 Read the output of the approved job

In [ ]:
ds_job_a = ds_client.jobs[JOB_A_NAME]
print(f"Status: {ds_job_a.status}")
print(f"Output files: {ds_job_a.output_paths}")

if ds_job_a.output_paths:
    print(f"\nResult:\n{ds_job_a.output_paths[0].read_text()}")

### 5.3 See the rejection reason for the rejected job

In [ ]:
ds_job_b = ds_client.jobs[JOB_B_NAME]
print(f"Status: {ds_job_b.status}")
print(f"Rejection reason: {ds_job_b.reason}")

---

## 6. Cleanup

Remove generated test files.

In [ ]:
!rm -rf data/PDFs data/manifest.csv data/readme.md